# 04 — Gold Star Schema

Goal: transform Silver events into a star schema.

```text
SILVER EVENTS
     |
     +--> DIM_USER
     +--> DIM_DATE
     |
     +--> FACT_EVENTS
```

In [1]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("medallion-gold-star")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [2]:
data = [
    ("e1","u1","purchase",10.0,"2026-01-01 10:00"),
    ("e2","u1","purchase",15.0,"2026-01-01 10:05"),
    ("e3","u2","refund",5.0,"2026-01-01 11:00"),
]

df = (
    spark.createDataFrame(data,["event_id","user_id","event_type","amount","event_time"])
    .withColumn("event_time",F.to_timestamp("event_time"))
    .withColumn("event_date",F.to_date("event_time"))
)
df.show()

+--------+-------+----------+------+-------------------+----------+
|event_id|user_id|event_type|amount|         event_time|event_date|
+--------+-------+----------+------+-------------------+----------+
|      e1|     u1|  purchase|  10.0|2026-01-01 10:00:00|2026-01-01|
|      e2|     u1|  purchase|  15.0|2026-01-01 10:05:00|2026-01-01|
|      e3|     u2|    refund|   5.0|2026-01-01 11:00:00|2026-01-01|
+--------+-------+----------+------+-------------------+----------+



## DIM_USER

In [3]:
dim_user = df.select("user_id").distinct()
dim_user.show()

+-------+
|user_id|
+-------+
|     u1|
|     u2|
+-------+



## DIM_DATE

In [4]:
dim_date = df.select("event_date").distinct()
dim_date.show()

+----------+
|event_date|
+----------+
|2026-01-01|
+----------+



## FACT_EVENTS

In [5]:
fact_events = df.select("event_id","user_id","event_date","event_type","amount")
fact_events.show()

+--------+-------+----------+----------+------+
|event_id|user_id|event_date|event_type|amount|
+--------+-------+----------+----------+------+
|      e1|     u1|2026-01-01|  purchase|  10.0|
|      e2|     u1|2026-01-01|  purchase|  15.0|
|      e3|     u2|2026-01-01|    refund|   5.0|
+--------+-------+----------+----------+------+

